# Проверка флага `geo_mismatch`

Расчёты для раздела 1.6 отчёта: отражает ли флаг устойчивое поведение игроков или распределён случайно. Окно — 90 дней после регистрации.

Данные по игрокам берутся запросом `sql/08_geo_mismatch_check.sql`, вероятность считается здесь.

In [2]:
import os
from pathlib import Path

import pandas as pd
from scipy.stats import binom
from sqlalchemy import create_engine, text

## Данные

База — та же, что собирает `scripts/load.sh` (по умолчанию `igaming`, другое имя задаётся переменной `DB`).

In [3]:
database = os.environ.get("DB", "igaming")
engine = create_engine(f"postgresql+psycopg:///{database}")

query = Path("../sql/08_geo_mismatch_check.sql").read_text(encoding="utf-8")
with engine.connect() as connection:
    players = pd.read_sql(text(query), connection)

players.head()

,player_id,total_sessions,mismatch_sessions,expected_by_chance,foreign_countries,first_session_mismatch,chance_probability
0,945,120,36,4.29,9,False,2.384947e-23
1,1763,30,15,1.07,7,False,1.881774e-14
2,5378,31,15,1.11,9,True,3.523868e-14
3,7694,30,10,1.07,5,True,5.353275e-08
4,6354,6,5,0.21,4,False,3.417196e-07


## Частота несовпадений

Доля сессий с `geo_mismatch` среди всех сессий за 90 дней. Это вероятность несовпадения в одной случайной сессии.

In [4]:
mismatch_rate = players["mismatch_sessions"].sum() / players["total_sessions"].sum()
print(f"Играющих игроков: {len(players)}")
print(f"Доля сессий с несовпадением: {mismatch_rate:.1%}")

Играющих игроков: 1063
Доля сессий с несовпадением: 10.2%


## Гипотеза 1. Переезд

Если игрок живёт не в заявленной стране, повторные несовпадения должны идти из одной и той же чужой страны.

In [8]:
mismatch_players = players[players["mismatch_sessions"] > 0]
repeated = mismatch_players[mismatch_players["mismatch_sessions"] >= 2]

print(f"Игроков с несовпадением: {len(mismatch_players)}")
print(f"Из них с одним несовпадением: {(mismatch_players['mismatch_sessions'] == 1).sum()}")
print(f"С 2+ несовпадениями: {len(repeated)}")
print(f"  все из разных стран: {(repeated['foreign_countries'] == repeated['mismatch_sessions']).sum()}")
print(f"  все из одной страны: {(repeated['foreign_countries'] == 1).sum()}")

Игроков с несовпадением: 1063
Из них с одним несовпадением: 862
С 2+ несовпадениями: 201
  все из разных стран: 158
  все из одной страны: 15


## Гипотеза 2. Неверная страна при регистрации

Если игроки указывают неверную страну при регистрации, несовпадение в первой сессии должно встречаться чаще, чем в сессиях в целом.

In [6]:
first_session_share = players["first_session_mismatch"].mean()
print(f"Первая сессия с несовпадением: {players['first_session_mismatch'].sum()} игроков ({first_session_share:.1%})")
print(f"Несовпадение в сессиях в целом: {mismatch_rate:.1%}")

Первая сессия с несовпадением: 260 игроков (24.5%)
Несовпадение в сессиях в целом: 10.2%


## Гипотеза 3. Систематическая игра из другой страны

Для каждого игрока считается вероятность получить столько несовпадений или больше случайно, если каждая сессия независимо даёт несовпадение с частотой `mismatch_rate` (биномиальное распределение).

Игрок считается подозрительным, если эта вероятность меньше 1 из 1 000. Граница строже, чем 99% в цели 1, потому что проверяется каждый игрок по отдельности: при границе 1 из 100 десятки игроков попали бы в список случайно.

In [7]:
mismatch_players = mismatch_players.assign(
    expected_by_chance=lambda frame: (frame["total_sessions"] * mismatch_rate).round(2),
    chance_probability=lambda frame: binom.sf(frame["mismatch_sessions"] - 1, frame["total_sessions"], mismatch_rate),
)

suspicious = mismatch_players[mismatch_players["chance_probability"] < 0.001].sort_values("chance_probability")
print(f"Подозрительных игроков: {len(suspicious)}")
suspicious

Подозрительных игроков: 5


,player_id,total_sessions,mismatch_sessions,expected_by_chance,foreign_countries,first_session_mismatch,chance_probability
0,945,120,36,12.23,9,False,1.718998e-09
1,1763,30,15,3.06,7,False,4.583118e-08
2,5378,31,15,3.16,9,True,8.037205e-08
4,6354,6,5,0.61,4,False,6.031747e-05
3,7694,30,10,3.06,5,True,5.283039e-04
